In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go 


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')


In [2]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
csv_file = list(uploaded.keys())[0]
df = pd.read_csv(csv_file)

df.head()

Saving walmart Retail Data.csv to walmart Retail Data.csv


,City,Customer Age,Customer Name,Customer Segment,Discount,Number of Records,Order Date,Order ID,Order Priority,Order Quantity,...,Profit,Region,Row ID,Sales,Ship Date,Ship Mode,Shipping Cost,State,Unit Price,Zip Code
0,McKeesport,NaN,Jessica Myrick,Small Business,0.10,1,2012-01-01,28774,High,32,...,-111.80,East,4031,180.36,2012-01-02,Regular Air,4.69,Pennsylvania,5.98,15131
1,Bowie,NaN,Matt Collister,Home Office,0.08,1,2012-01-01,13729,Not Specified,9,...,-342.91,East,1914,872.48,2012-01-03,Express Air,35.00,Maryland,95.99,20715
2,Napa,NaN,Alan Schoenberger,Corporate,0.00,1,2012-01-02,37537,Low,4,...,-193.08,West,5272,1239.06,2012-01-02,Delivery Truck,48.80,California,291.73,94559
3,Montebello,NaN,Elizabeth Moffitt,Consumer,0.08,1,2012-01-02,44069,Critical,43,...,247.79,West,6225,614.80,2012-01-02,Regular Air,1.97,California,15.04,90640
4,Napa,NaN,Alan Schoenberger,Corporate,0.07,1,2012-01-02,37537,Low,43,...,-1049.85,West,5273,4083.19,2012-01-04,Delivery Truck,45.00,California,100.98,94559


## 3. Data Preprocessing & ETL Transformation (Full Pipeline)

In [ ]:

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])


cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(include=['int64','float64']).columns


df['Customer Age'].fillna(df['Customer Age'].median(), inplace=True)
df['Product Base Margin'].fillna(df['Product Base Margin'].median(), inplace=True)


for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)


initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - len(df)} duplicate rows.")


for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)

df.reset_index(drop=True, inplace=True)


print("\nMissing values per column after cleaning:\n", df.isnull().sum().loc[df.isnull().sum() > 0])
print("Duplicate rows count after cleaning:", df.duplicated().sum())
df.info()


Removed 0 duplicate rows.

Missing values per column after cleaning:
 Series([], dtype: int64)
Duplicate rows count after cleaning: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8399 entries, 0 to 8398
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   City                  8399 non-null   object        
 1   Customer Age          8399 non-null   float64       
 2   Customer Name         8399 non-null   object        
 3   Customer Segment      8399 non-null   object        
 4   Discount              8399 non-null   float64       
 5   Number of Records     8399 non-null   int64         
 6   Order Date            8399 non-null   datetime64[ns]
 7   Order ID              8399 non-null   int64         
 8   Order Priority        8399 non-null   object        
 9   Order Quantity        8399 non-null   int64         
 10  Product Base Margin   8399 non-null   float64       
 11  

In [ ]:

df['Ship_Duration_Days'] = (df['Ship Date'] - df['Order Date']).dt.days
print("New Feature Check (Ship Duration):")
print(df[['Order Date', 'Ship Date', 'Ship_Duration_Days']].head())


df['Order_Year'] = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.month
df['Order_Month_Year'] = df['Order Date'].dt.strftime('%Y-%m')


df['Profit_Margin'] = df['Profit'] / df['Sales']


df['Profit_Label'] = df['Profit'].apply(lambda x: 1 if x > 0 else 0)

print("\nTarget Variable Distribution:")
print(df['Profit_Label'].value_counts())


New Feature Check (Ship Duration):
  Order Date  Ship Date  Ship_Duration_Days
0 2012-01-01 2012-01-02                   1
1 2012-01-01 2012-01-03                   2
2 2012-01-02 2012-01-02                   0
3 2012-01-02 2012-01-02                   0
4 2012-01-02 2012-01-04                   2

Target Variable Distribution:
Profit_Label
0    4264
1    4135
Name: count, dtype: int64


## 5. Exploratory Data Analysis (EDA) and Visualization

In [5]:
monthly_sales = df.groupby('Order_Month_Year')['Sales'].sum().reset_index()

fig = px.line(
 monthly_sales,
 x='Order_Month_Year',
 y='Sales',
 title='Monthly Sales Trend',
 labels={'Order_Month_Year':'Month-Year', 'Sales':'Total Sales'},
 markers=True
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()


In [6]:
profit_by_category = df.groupby('Product Category')['Profit'].sum().reset_index()

fig = px.bar(
 profit_by_category,
 x='Product Category',
 y='Profit',
 text='Profit',
 title='Total Profit by Product Category',
 labels={'Profit':'Total Profit'},
 template='plotly_white',
 color='Product Category',
 color_discrete_map={'Furniture': 'orange', 'Office Supplies': 'lightskyblue', 'Technology': 'lightcoral'}
)

fig.update_traces(texttemplate='%{text:.2s}', textposition='outside', width=0.4)
fig.update_layout(yaxis=dict(title='Total Profit'))

fig.show()


In [7]:
loss_df = df[df['Profit'] < 0]

loss_summary = loss_df.groupby('Product Sub-Category')['Profit_Label'].count().reset_index()
loss_summary = loss_summary.sort_values(by='Profit_Label', ascending=False)

fig = px.bar(
 loss_summary,
 x='Product Sub-Category',
 y='Profit_Label',
 text='Profit_Label',
 title='Loss-Making Orders by Product Sub-Category',
 labels={'Profit_Label':'Number of Loss Orders', 'Product Sub-Category':'Sub-Category'},
 template='plotly_white',
 color='Profit_Label',
 color_continuous_scale='teal'
)


fig.update_traces(textposition='outside', textfont_size=12)
fig.update_layout(
 yaxis=dict(title='Number of Loss Orders', automargin=True),
 xaxis=dict(title='Product Sub-Category', tickangle=-45, automargin=True)
)

fig.show()


In [ ]:

y = df['Profit_Label']

X = df[['Order Quantity','Discount','Unit Price','Product Base Margin',
       'Region','Product Category','Product Sub-Category',
       'Customer Segment','Order Priority',
       'Order_Year','Order_Month', 'Ship_Duration_Days']]


priority_mapping = {'Low':1, 'Medium':2, 'High':3, 'Critical':4, 'Not Specified':0}
X['Order Priority'] = X['Order Priority'].map(priority_mapping)


nominal_cols = ['Region','Product Category','Product Sub-Category','Customer Segment']
X_encoded = pd.get_dummies(X, columns=nominal_cols, drop_first=True)


X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

print("Data Preparation Complete. Shape of encoded features:", X_encoded.shape)


Data Preparation Complete. Shape of encoded features: (8399, 32)


In [ ]:

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)


y_pred_rf = rf_model.predict(X_test)
pred_probs_rf = rf_model.predict_proba(X_test)[:, 1]


rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_report = classification_report(y_test, y_pred_rf, output_dict=True)

print(f"Random Forest Accuracy: {rf_accuracy:.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf, target_names=['Loss (0)', 'Profit (1)']))

cm_rf = confusion_matrix(y_test, y_pred_rf)
labels = ['Loss', 'Profit']
z_text = [[str(y) for y in x] for x in cm_rf]

fig_cm_rf = go.Figure(data=go.Heatmap(z=cm_rf, x=labels, y=labels, hoverinfo='z', text=z_text, texttemplate="%{text}", colorscale='Blues', showscale=True))
fig_cm_rf.update_layout(title='Random Forest Confusion Matrix', xaxis=dict(title='Predicted Label'), yaxis=dict(title='Actual Label'), width=600, height=500)
fig_cm_rf.show()


importances = rf_model.feature_importances_
feature_names = X_encoded.columns
feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False)
top10 = feat_imp.head(10)

fig_feat = px.bar(
    top10, x='Importance', y='Feature', orientation='h', text='Importance',
    title='Random Forest Top 10 Feature Importance'
)
fig_feat.update_layout(yaxis=dict(autorange="reversed"))
fig_feat.update_traces(texttemplate='%{text:.2f}', textposition='outside', marker_color='skyblue')
fig_feat.show()


test_df = df.loc[X_test.index].copy()
test_df['Actual_Label'] = y_test
test_df['Predicted_Label'] = y_pred_rf
test_df['Predicted_Prob'] = pred_probs_rf

test_df.sort_values(by='Profit', ascending=False, inplace=True)
profit_orders = test_df[test_df['Profit'] > 0].head(10)
loss_orders = test_df[test_df['Profit'] < 0].head(10)


Random Forest Accuracy: 0.86

Classification Report:
               precision    recall  f1-score   support

    Loss (0)       0.86      0.88      0.87       853
  Profit (1)       0.87      0.85      0.86       827

    accuracy                           0.86      1680
   macro avg       0.86      0.86      0.86      1680
weighted avg       0.86      0.86      0.86      1680



In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


lr_model = LogisticRegression(random_state=42, solver='liblinear')
lr_model.fit(X_train_scaled, y_train)


y_pred_lr = lr_model.predict(X_test_scaled)


lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_report = classification_report(y_test, y_pred_lr, output_dict=True)

print(f"Logistic Regression Accuracy: {lr_accuracy:.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr, target_names=['Loss (0)', 'Profit (1)']))


cm_lr = confusion_matrix(y_test, y_pred_lr)
labels = ['Loss', 'Profit']
z_text = [[str(y) for y in x] for x in cm_lr]

fig_cm_lr = go.Figure(data=go.Heatmap(z=cm_lr, x=labels, y=labels, hoverinfo='z', text=z_text, texttemplate="%{text}", colorscale='Reds', showscale=True))
fig_cm_lr.update_layout(title='Logistic Regression Confusion Matrix', xaxis=dict(title='Predicted Label'), yaxis=dict(title='Actual Label'), width=600, height=500)
fig_cm_lr.show()


Logistic Regression Accuracy: 0.68

Classification Report:
               precision    recall  f1-score   support

    Loss (0)       0.68      0.70      0.69       853
  Profit (1)       0.68      0.66      0.67       827

    accuracy                           0.68      1680
   macro avg       0.68      0.68      0.68      1680
weighted avg       0.68      0.68      0.68      1680



In [ ]:
comp_df = pd.DataFrame({
    'Model': ['Random Forest', 'Logistic Regression'],
    'Accuracy': [rf_accuracy, lr_accuracy],
    'Precision (Profit)': [rf_report['1']['precision'], lr_report['1']['precision']],
    'Recall (Profit)': [rf_report['1']['recall'], lr_report['1']['recall']],
    'F1-Score (Profit)': [rf_report['1']['f1-score'], lr_report['1']['f1-score']]
})

comp_df = comp_df.set_index('Model').T
print(comp_df)

fig_comp = px.bar(comp_df.T, x=comp_df.index, y=comp_df.columns, barmode='group',
                  title='Model Comparison: Key Performance Metrics',
                  labels={'value': 'Score', 'variable': 'Metric', 'Model': 'Model'})

fig_comp.update_layout(yaxis=dict(range=[0.5, 1.0])) 
fig_comp.show()


Model               Random Forest  Logistic Regression
Accuracy                 0.863095             0.680357
Precision (Profit)       0.868974             0.681704
Recall (Profit)          0.850060             0.657799
F1-Score (Profit)        0.859413             0.669538


In [16]:
print("Value counts for Actual_Label in Top 10 Loss Orders:\n", loss_orders['Actual_Label'].value_counts())
print("\nValue counts for Predicted_Label in Top 10 Loss Orders:\n", loss_orders['Predicted_Label'].value_counts())

Value counts for Actual_Label in Top 10 Loss Orders:
 Actual_Label
0    10
Name: count, dtype: int64

Value counts for Predicted_Label in Top 10 Loss Orders:
 Predicted_Label
0    7
1    3
Name: count, dtype: int64


In [ ]:
def plot_actual_vs_predicted_dots(df_data, title_text, filter_predicted_profit_on_loss=False):

    plot_df = df_data[['Order ID', 'Actual_Label', 'Predicted_Label', 'Profit']].melt(
        id_vars=['Order ID', 'Profit'],
        value_vars=['Actual_Label', 'Predicted_Label'],
        var_name='Label_Type',
        value_name='Label'
    )

    plot_df['Label_Text'] = plot_df['Label'].map({1: 'Profit (1)', 0: 'Loss (0)'})


    plot_df['Combined_Label'] = plot_df['Label_Type'].str.replace('_Label', '') + ' ' + plot_df['Label_Text']


    if filter_predicted_profit_on_loss:

        plot_df = plot_df[~((plot_df['Label_Type'] == 'Predicted_Label') & (plot_df['Label'] == 1))]


    size_map = {'Actual_Label': 10, 'Predicted_Label': 5}
    plot_df['Dot_Size'] = plot_df['Label_Type'].map(size_map)

    fig = px.scatter(
        plot_df,
        x='Order ID',
        y='Profit',
        color='Combined_Label',
        symbol='Label_Type',
        size='Dot_Size', 
        size_max=15,
        hover_data=['Profit', 'Label_Text'],
        title=title_text,
        labels={'Profit': 'Actual Profit ($)', 'Order ID': 'Order ID'}
    )

    fig.update_layout(
        title={'x':0.5, 'xanchor': 'center'},
        xaxis={'tickangle': -45},
        legend_title='Label Status',
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()


plot_actual_vs_predicted_dots(profit_orders, "Actual vs Predicted Labels (Top 10 Profit Orders)")
plot_actual_vs_predicted_dots(loss_orders, "Actual vs Predicted Labels (Top 10 Loss Orders - Most Negative Profit)", filter_predicted_profit_on_loss=True)


print("===== Top 10 Profit Orders ====")
print(profit_orders[['Order ID','Order Quantity','Unit Price','Actual_Label','Predicted_Label','Predicted_Prob']])

print("\n===== Top 10 Loss Orders (Most Negative Profit) ====")
print(loss_orders[['Order ID','Order Quantity','Unit Price','Actual_Label','Predicted_Label','Predicted_Prob']])

===== Top 10 Profit Orders ====
      Order ID  Order Quantity  Unit Price  Actual_Label  Predicted_Label  \
5333     34663              38      808.49             1                1   
1642     41728              28      896.99             1                1   
1762     50626              42      599.99             1                1   
61        7203              25      896.99             1                1   
6080     29795              49      387.99             1                1   
4733     28550              44      500.97             1                1   
5403      6596              36      500.98             1                1   
4065     55653              34      500.98             1                1   
6056     37252              34      880.98             1                1   
213      21383              31      574.74             1                1   

      Predicted_Prob  
5333            0.93  
1642            0.90  
1762            0.86  
61              0.86  
6080 